# Example-04: Initialization from harmonics

In [1]:
# Import

import epics
import numpy
import torch

import sys
sys.path.append('..')

from harmonica.util import pv_make
from harmonica.window import Window
from harmonica.data import Data

import matplotlib.pyplot as plt
from time import sleep

torch.set_printoptions(precision=12, sci_mode=True)
print(torch.cuda.is_available())
print(torch.get_num_threads())

True
16


In [2]:
# Set data type and device

dtype = torch.float64
device = torch.device('cpu')

In [3]:
# This initialization can be used to generate test data from given harmonics
# Another option is to create empty instance and set data and work containers
# Or use from_data (classmethod)

# Set window

w = Window(length=8192, name='cosine_window', order=1.0, dtype=dtype, device=device)
print(w)

# Generate harmonics (generate two signals with three harmonics in each signal)

# A harmonic can be generated with Data.make_harmonic (staticmethod)

m, f, c, s = 0.5, [0.10, 0.20, 0.30], [5.0, 0.5, 0.1], [0.1, 0.5, 1.0]
h1 = m*torch.ones(w.length, dtype=dtype, device=device)
for f, c, s in zip(f, c, s):
    h1 += Data.make_harmonic(w.length, f, c=c, s=s, dtype=dtype, device=device)
    
m, f, c, s = 1.0, [0.12, 0.23, 0.34], [5.5, 1.5, 4.1], [0.2, 0.6, 4.0]
h2 = m*torch.ones(w.length, dtype=dtype, device=device)
for f, c, s in zip(f, c, s):
    h2 += Data.make_harmonic(w.length, f, c=c, s=s, dtype=dtype, device=device)
    
# Set TbT

d = Data.from_data(w, torch.stack([h1, h2]))
print(d)

# Check data frequencies (largest bin in FFT spectrum)

print(torch.argmax(torch.abs(torch.fft.fft(d.data, 1000)), 1).to(dtype).cpu().div_(1000).numpy())

# Check amplitudes

t = 2.0*numpy.pi*torch.linspace(0, d.length-1, d.length, dtype=dtype, device=device)
f = torch.tensor([[0.10, 0.20, 0.30], [0.12, 0.23, 0.34]], dtype=dtype, device=device).reshape(d.size, -1, 1)
for i in range(d.size):
    print((2*torch.sum(d.window.window*d[i]*torch.cos(t*f[i]), 1)/d.length).cpu().numpy())
    print((2*torch.sum(d.window.window*d[i]*torch.sin(t*f[i]), 1)/d.length).cpu().numpy())
    
# Clean

del w
del h1, h2
del d
if device != torch.device('cpu'):
    torch.cuda.synchronize()
    torch.cuda.empty_cache()

Window(8192, 'cosine_window', 1.0)
Data(2, Window(8192, 'cosine_window', 1.0))
[0.1  0.34]
[5.  0.5 0.1]
[0.1 0.5 1. ]
[5.5 1.5 4.1]
[0.2 0.6 4. ]
